In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text
import seaborn as sns
from scipy import stats
import configparser
sns.set_theme()

In [ ]:
filepath = os.getcwd()

config = configparser.ConfigParser()
config.read('.secrets.ini')

bincols = ['1hrControl', '1hrAfter', '2hrControl', '2hrAfter', '12hrControl', '12hrAfter', '7dayControl', '7dayAfter']

# Set up SQL Connection
url = URL.create("postgresql",
                 username = config['sql-login']['username'], password=config['sql-login']['password'],
                 database=config['sql-login']['database'], host=config['sql-login']['host']
                 )
dbconn1 = create_engine(url, execution_options={'stream_results':False}).connect()

eqs = pd.read_sql('select * from earthquakes_processed', con=dbconn1)

In [ ]:
# Crowdsourced crash counts in 2-hour earthquake periods
countsEQ = pd.read_csv('WazeCrashCounts_ByEarthquake.txt')

countsEQ = countsEQ.merge(eqs, on=['id'])
fig, ax = plt.subplots(1, 2, sharex=False, sharey=False,  figsize=(12, 5), layout='constrained')

# Create scatterplot of crashes in control vs. earthquake periods
# Earthquake magnitude is denoted by point size and color
countsEQ.plot.scatter(ax = ax[0], x ='2hrControl', y='2hrAfter', c='mag', colorbar=False, s=countsEQ['mag']*40*((countsEQ['mag']-countsEQ['mag'].min())/(countsEQ['mag'].max()-countsEQ['mag'].min())))
for collection in ax[0].collections:
    collection.set_edgecolor('black')
    collection.set_linewidth(1)

# Add one-to-one line
nums = np.linspace(0, max(countsEQ['2hrControl'].max(), countsEQ['2hrAfter'].max()), num=100)
oneone = pd.DataFrame({'x': nums, 'y': nums})
oneone.plot(x = 'x', y='y', ax = ax[0], label = "One-to-One Line")

# Add linear fit line
z = np.polyfit(countsEQ['2hrControl'], countsEQ['2hrAfter'], 1)
p = np.poly1d(z)
trendline = pd.DataFrame({'x': nums, 'y': p(nums)})
trendline.plot(x = 'x', y='y', ax = ax[0], style="r--", label = "Linear Fit")

ax[0].set_xlabel(f'Crashes in 2 Hour Control Periods')
ax[0].set_ylabel(f'Crashes in 2 Hour Earthquake Periods')
ax[0].set_title('Waze Crash Reports')
ax[0].get_legend().remove()

# Calculate R-squared
y_pred = p(countsEQ['2hrControl']) # Predicted values
y_bar = np.mean(countsEQ['2hrAfter']) # Mean of actual values
SStot = np.sum((countsEQ['2hrAfter'] - y_bar) ** 2) # Total Sum of Squares
SSres = np.sum((countsEQ['2hrAfter'] - y_pred) ** 2) # Residual Sum of Squares
r_squared = 1 - (SSres / SStot) # R-squared
ax[0].text(300, 350, "y=%.2fx+(%.2f), R2=%.2f"%(z[0],z[1], r_squared), rotation=38)

y_pred1 = p(countsEQ['2hrControl'])
# Mean of actual values
y_bar1 = np.mean(countsEQ['2hrAfter'])
# Residuals (actual - predicted)
res1 = countsEQ['2hrAfter'] - y_pred1
# Total Sum of Squares
SStot1 = np.sum((countsEQ['2hrAfter'] - y_bar1) ** 2)
# Residual Sum of Squares
SSres1 = np.sum(res1 ** 2)
# Calculate R-squared
r_squared1 = 1 - (SSres1 / SStot1)

y_pred2 = countsEQ['2hrControl']
# Mean of actual values
y_bar2 = np.mean(countsEQ['2hrAfter'])
# Residuals (actual - predicted)
res2 = countsEQ['2hrAfter'] - y_pred2
# Total Sum of Squares
SStot2 = np.sum((countsEQ['2hrAfter'] - y_bar2) ** 2)
# Residual Sum of Squares
SSres2 = np.sum(res2 ** 2)
# Calculate R-squared
r_squared2 = 1 - (SSres2 / SStot2)

# Police-reported crash counts in 2-hour earthquake periods
countsEQ = pd.read_csv('CHPCounts_ByEarthquake.txt')
countsEQ = countsEQ.merge(eqs, on=['id'])
countsEQ['Earthquake Magnitude'] = countsEQ['mag']

# Create scatterplot of crashes in control vs. earthquake periods
countsEQ.plot.scatter(ax = ax[1],  x ='2hrControl', y='2hrAfter', colorbar=False, c='Earthquake Magnitude', s=1+countsEQ['Earthquake Magnitude']*40*((countsEQ['mag']-countsEQ['mag'].min())/(countsEQ['mag'].max()-countsEQ['mag'].min())))
for collection in ax[1].collections:
    collection.set_edgecolor('black')
    collection.set_linewidth(1)

# Add one-to-one line
nums = np.linspace(0, max(countsEQ['2hrControl'].max(), countsEQ['2hrAfter'].max()), num=100)
oneone = pd.DataFrame({'x': nums, 'y': nums})
oneone.plot(x = 'x', y='y', ax = ax[1], label = "One-to-One Line")

# Add linear fit line
z = np.polyfit(countsEQ['2hrControl'], countsEQ['2hrAfter'], 1)
p = np.poly1d(z)
trendline = pd.DataFrame({'x': nums, 'y': p(nums)})
trendline.plot(x = 'x', y='y', ax = ax[1], style="r--", label = "Linear Fit")

# Calculate R-squared
y_pred = p(countsEQ['2hrControl']) # Predicted values
y_bar = np.mean(countsEQ['2hrAfter']) # Mean of actual values
SStot = np.sum((countsEQ['2hrAfter'] - y_bar) ** 2) # Total Sum of Squares
SSres = np.sum((countsEQ['2hrAfter'] - y_pred) ** 2) # Residual Sum of Squares
r_squared = 1 - (SSres / SStot) # R-squared
ax[1].text(57, 49.25, "y=%.2fx+(%.2f), R2=%.2f"%(z[0],z[1], r_squared), rotation=35, label = "Linear Fit")

ax[1].set_xlabel(f'Crashes in 2 Hour Control Periods')
ax[1].set_ylabel('')
ax[1].set_title('CHP Crash Reports')
ax[1].get_legend().remove()

h1, l1 = ax[0].get_legend_handles_labels()
mag = countsEQ['mag'].values

# colormap & normalization
cmap = plt.cm.gist_heat
norm = mpl.colors.Normalize(vmin=mag.min(), vmax=mag.max())

# size mapping
def size_map(m):
    denom = (m.max() - m.min()) if (m.max() != m.min()) else 1.0
    return 1 + m * 40 * ((m - m.min()) / denom)

sizes = size_map(mag)

rep_mags = np.array([4, 5, 6, 7])
rep_sizes = size_map(rep_mags)
rep_colors = [cmap(norm(m)) for m in rep_mags]

legend_handles = []
legend_labels = []
for mval, sz, col in zip(rep_mags, rep_sizes, rep_colors):
    # make a scatter handle with the same marker, color, and size
    handle = ax[1].scatter([], [], s=sz, c=[col], edgecolor='k', alpha=0.85)
    legend_handles.append(handle)
    legend_labels.append(f"{mval:.1f}")

# Create empty handles for spacing
empty = plt.Line2D([], [], linestyle='none', marker=None, alpha=0.0) 
combined_handles = [legend_handles[0], empty, legend_handles[1], h1[0], legend_handles[2], h1[1], legend_handles[3], empty]
combined_labels = [legend_labels[0], '', legend_labels[1], l1[0], legend_labels[2], l1[1], legend_labels[3], '']
legend0 = fig.legend(legend_handles, legend_labels, ncol=4,
                   columnspacing=1.2, handletextpad=0.6, borderpad = .6,
                   labelspacing=1, frameon=True, bbox_to_anchor=(.41, 0), loc = 9, title='Earthquake Magnitude (Mw)')
legend1 = fig.legend(h1, l1, ncol=1,
                   columnspacing=1.2, handletextpad=0.6, borderpad = .65,
                   labelspacing=1, frameon=True, bbox_to_anchor=(.41+.21, 0), loc = 9)
                   
# Add a continuous colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.show()

In [ ]:
# Perform F-tests

y_pred3 = p(countsEQ['2hrControl'])
# Mean of actual values
y_bar3 = np.mean(countsEQ['2hrAfter'])
# Residuals (actual - predicted)
res3 = countsEQ['2hrAfter'] - y_pred3
# Total Sum of Squares
SStot3 = np.sum((countsEQ['2hrAfter'] - y_bar3) ** 2)
# Residual Sum of Squares
SSres3 = np.sum(res3 ** 2)
# Calculate R-squared
r_squared3 = 1 - (SSres3 / SStot3)
print(r_squared3)

y_pred4 = countsEQ['2hrControl']
# Mean of actual values
y_bar4 = np.mean(countsEQ['2hrAfter'])
# Residuals (actual - predicted)
res4 = countsEQ['2hrAfter'] - y_pred4
# Total Sum of Squares
SStot4 = np.sum((countsEQ['2hrAfter'] - y_bar4) ** 2)
# Residual Sum of Squares
SSres4 = np.sum(res4 ** 2)
# Calculate R-squared
r_squared4 = 1 - (SSres4 / SStot4)
print(r_squared4)
print(stats.f_oneway(res1, res2))
print(stats.f_oneway(res3, res4))